## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <assert.h>

#define MAXN 1029

/* ================= 动态数组================= */
typedef struct {
    int *data;
    int size;
    int cap;
} IntVec;

void vecInit(IntVec *v) {
    v->size = 0;
    v->cap = 16;
    v->data = (int *)malloc(sizeof(int) * v->cap);
    if (v->data == NULL) exit(1);
}

void vecPush(IntVec *v, int x) {
    if (v->size >= v->cap) {
        v->cap *= 2;
        int *newData = (int *)realloc(v->data, sizeof(int) * v->cap);
        if (newData == NULL) exit(1);
        v->data = newData;
    }
    v->data[v->size++] = x;
}

void vecFree(IntVec *v) {
    free(v->data);
    v->data = NULL;
    v->size = 0;
    v->cap = 0;
}

/* ================= 快读整数 ================= */
int readInt(void) {
    int x = 0;
    int ch = getchar();

    while (ch < '0' || ch > '9') ch = getchar();
    while (ch >= '0' && ch <= '9') {
        x = x * 10 + (ch - '0');
        ch = getchar();
    }
    return x;
}

/* ================= 全局变量 ================= */
int n;
int A, B;
int lowbitLen;

int p[MAXN];      // 当前排列：p[i] 表示位置 i 上的值
int pos[MAXN];    // 反向位置：pos[x] 表示值 x 当前在哪个位置
IntVec answer;    // 记录所有操作

/* 重新维护 pos 数组 */
void rebuildPos(void) {
    for (int i = 0; i < n; ++i) {
        pos[p[i]] = i;
    }
}

/* 操作 0：交换排列中所有值 A 和 B */
void useSwapMagic(void) {
    vecPush(&answer, 0);

    for (int i = 0; i < n; ++i) {
        if (p[i] == A) p[i] = B;
        else if (p[i] == B) p[i] = A;
    }

    rebuildPos();
}

/* 操作 2 v：所有值加 v，然后对 n 取模 */
void useAddMagic(int v) {
    v %= n;
    if (v < 0) v += n;
    if (v == 0) return;

    vecPush(&answer, v);

    for (int i = 0; i < n; ++i) {
        p[i] = (p[i] + v) % n;
    }

    rebuildPos();
}

/* 操作 1 v：所有值异或 v */
void useXorMagic(int v) {
    if (v == 0) return;

    vecPush(&answer, -v);

    for (int i = 0; i < n; ++i) {
        p[i] ^= v;
    }

    rebuildPos();
}

/* 根据 x、y 计算一组辅助位置 px、py */
void getPairPos(int x, int y, int *px, int *py) {
    int delta = (y - x + n - lowbitLen + n) % n;
    *px = 0;
    *py = 0;

    for (int step = n / 2; step >= 2 * lowbitLen; step >>= 1) {
        if (delta >= step) {
            delta -= step;
            *py += step / 2;
        } else {
            *px += step / 2;
        }
    }

    *px += n / 2;
    *px += (x & (lowbitLen - 1));
    *py += (x & (lowbitLen - 1));
}

void applySwap(int x, int y);

/*
   利用题目给定的三种 magic 操作，间接实现“交换值 x 和 y”的效果。
   注意：这里不是普通 swap，而是通过若干 add/xor/swapMagic 操作模拟出来。
*/
void applySwap(int x, int y) {
    /*
       如果 x 和 y 所在的 lowbitLen 块奇偶性相同，
       就找一个中间点 mid，使用三次交换把它转化成可处理情况。
    */
    if ((x / lowbitLen) % 2 == (y / lowbitLen) % 2) {
        int mid;

        if ((x / lowbitLen) % 2 == 0) {
            mid = (x & (lowbitLen - 1)) + lowbitLen;
        } else {
            mid = (x & (lowbitLen - 1));
        }

        applySwap(x, mid);
        applySwap(y, mid);
        applySwap(x, mid);
        return;
    }

    int pA, pB, pX, pY;
    getPairPos(A, B, &pA, &pB);
    getPairPos(x, y, &pX, &pY);

    /* 下面这一串操作对应原 C++ 标答，不能随便改顺序 */
    useAddMagic((pX - x + n) % n);
    useXorMagic(pX ^ pA);
    useAddMagic((A - pA + n) % n);

    useSwapMagic();

    useAddMagic((pA - A + n) % n);
    useXorMagic(pX ^ pA);
    useAddMagic((x - pX + n) % n);
}

/* ================= Permutation 结构：替代 C++ struct Permutation ================= */
typedef struct {
    int val[MAXN];
    int len;
    IntVec ops;
} Permutation;

void permInit(Permutation *perm, int len) {
    perm->len = len;
    vecInit(&perm->ops);
}

/*
   递归构造低位重排所需要的操作序列。
   返回 1 表示可以构造，返回 0 表示无法构造。
*/
int permBuild(Permutation *perm) {
    if (perm->len == 1) return 1;

    Permutation leftPart;
    Permutation rightPart;
    permInit(&leftPart, perm->len / 2);
    permInit(&rightPart, perm->len / 2);

    /*
       把当前排列按照偶数位置、奇数位置拆成两半。
       同时每个值除以 2，相当于去掉最低位。
    */
    for (int i = 0; i < perm->len / 2; ++i) {
        leftPart.val[i] = perm->val[i * 2] / 2;
        rightPart.val[i] = perm->val[i * 2 + 1] / 2;
    }

    if (!permBuild(&leftPart) || !permBuild(&rightPart)) {
        vecFree(&leftPart.ops);
        vecFree(&rightPart.ops);
        return 0;
    }

    /* 如果首元素最低位为 1，需要先补一个操作 */
    if (perm->val[0] & 1) {
        vecPush(&perm->ops, perm->len == 2 ? 1 : -1);
    }

    int curXorL = 0;
    for (int i = 0; i < leftPart.ops.size; ++i) {
        int x = leftPart.ops.data[i];

        if (x > 0) {
            vecPush(&perm->ops, -1);
            vecPush(&perm->ops, 1);
        } else {
            vecPush(&perm->ops, x * 2);
            curXorL ^= (-x) * 2;
        }
    }

    if (curXorL) {
        vecPush(&perm->ops, -curXorL);
    }

    int curXorR = 0;
    for (int i = 0; i < rightPart.ops.size; ++i) {
        int x = rightPart.ops.data[i];

        if (x > 0) {
            vecPush(&perm->ops, 1);
            vecPush(&perm->ops, -1);
        } else {
            vecPush(&perm->ops, x * 2);
            curXorR ^= (-x) * 2;
        }
    }

    /* 两边构造出来的异或状态必须兼容，否则无解 */
    if ((curXorR & (perm->len / 2)) != (curXorL & (perm->len / 2))) {
        vecFree(&leftPart.ops);
        vecFree(&rightPart.ops);
        return 0;
    }

    if (curXorL >= perm->len / 2) curXorL -= perm->len / 2;
    if (curXorR >= perm->len / 2) curXorR -= perm->len / 2;

    if (curXorL != curXorR) {
        vecFree(&leftPart.ops);
        vecFree(&rightPart.ops);
        return 0;
    }

    /*
       合并连续的 xor 操作。
       原理：连续执行 xor a 和 xor b，等价于 xor (a ^ b)。
    */
    IntVec merged;
    vecInit(&merged);

    for (int i = 0; i < perm->ops.size; ++i) {
        int x = perm->ops.data[i];

        if (merged.size == 0) {
            vecPush(&merged, x);
        } else {
            int last = merged.data[merged.size - 1];

            if (x < 0 && last < 0) {
                int newOp = -((-last) ^ (-x));
                if (newOp == 0) {
                    merged.size--;
                } else {
                    merged.data[merged.size - 1] = newOp;
                }
            } else {
                vecPush(&merged, x);
            }
        }
    }

    /* 用压缩后的操作序列替换原操作序列 */
    vecFree(&perm->ops);
    perm->ops = merged;

    vecFree(&leftPart.ops);
    vecFree(&rightPart.ops);
    return 1;
}

int cmpInt(const void *a, const void *b) {
    int x = *(const int *)a;
    int y = *(const int *)b;
    return (x > y) - (x < y);
}

int main(void) {
    vecInit(&answer);

    n = readInt();
    A = readInt();
    B = readInt();

    for (int i = 0; i < n; ++i) {
        p[i] = readInt();
    }

    rebuildPos();

    /*
       lowbitLen 取 A-B 的最低位 1 对应的长度。
       如果 A == B，则 lowbitLen = n。
    */
    lowbitLen = (A - B + n) % n;
    lowbitLen &= -lowbitLen;
    if (lowbitLen == 0) lowbitLen = n;

    /* 先处理低 lowbitLen 位上的排列问题 */
    if (lowbitLen > 1) {
        Permutation base;
        permInit(&base, lowbitLen);

        for (int i = 0; i < n; ++i) {
            base.val[i] = p[i] & (lowbitLen - 1);
        }

        if (!permBuild(&base)) {
            printf("-1\n");
            vecFree(&base.ops);
            vecFree(&answer);
            return 0;
        }

        for (int i = 0; i < base.ops.size; ++i) {
            int x = base.ops.data[i];
            if (x > 0) useAddMagic(x);
            else useXorMagic(-x);
        }

        vecFree(&base.ops);
    }

    /* 再按每个余数类分别检查并归位 */
    for (int rem = 0; rem < lowbitLen; ++rem) {
        int vec[MAXN];
        int cnt = 0;

        for (int j = rem; j < n; j += lowbitLen) {
            vec[cnt++] = p[j];
        }

        qsort(vec, cnt, sizeof(int), cmpInt);

        int ok = 1;
        int ptr = 0;
        for (int j = rem; j < n; j += lowbitLen) {
            if (vec[ptr] != j) {
                ok = 0;
                break;
            }
            ++ptr;
        }

        if (!ok) {
            printf("-1\n");
            vecFree(&answer);
            return 0;
        }

        /* 把这个余数类中的元素一个个放回正确位置 */
        for (int j = rem; j < n; j += lowbitLen) {
            if (p[j] != j) {
                applySwap(j, p[j]);
            }
        }
    }

    /* 调试保护：最终排列必须变成 0,1,2,...,n-1 */
    for (int i = 0; i < n; ++i) {
        assert(p[i] == i);
    }

    /* 输出操作数量和操作本身 */
    printf("%d\n", answer.size);
    for (int i = 0; i < answer.size; ++i) {
        int x = answer.data[i];

        if (x == 0) {
            printf("0\n");
        } else if (x < 0) {
            printf("1 %d\n", -x);   // xor 操作
        } else {
            printf("2 %d\n", x);    // add 操作
        }
    }

    vecFree(&answer);
    return 0;
}

## B 长跑

In [ ]:
import sys
from collections import deque

def solve():
    data = list(map(int, sys.stdin.buffer.read().split()))
    idx = 0
    ans = []
    INF = 10**18

    while idx < len(data):
        N, L, Maxn, S = data[idx], data[idx + 1], data[idx + 2], data[idx + 3]
        idx += 4

        # 同一位置可能有多个补给站，只保留最便宜的
        pos_cost = {}
        for _ in range(N):
            p, c = data[idx], data[idx + 1]
            idx += 2

            # 到了终点或终点之后的补给没有意义
            if p >= L:
                continue

            if p not in pos_cost or c < pos_cost[p]:
                pos_cost[p] = c

        # 起点直接到终点
        if L <= Maxn:
            ans.append("Yes")
            continue

        stations = sorted(pos_cost.items())   # (position, cost)
        m = len(stations)

        if m == 0:
            ans.append("No")
            continue

        dp = [INF] * m
        q = deque()   # 存下标，且对应 dp 单调递增

        for i in range(m):
            p_i, c_i = stations[i]

            # 把窗口外的点弹掉：要求 p_i - p_j <= Maxn
            while q and p_i - stations[q[0]][0] > Maxn:
                q.popleft()

            # 从起点直接到当前补给站
            best = INF
            if p_i <= Maxn:
                best = c_i

            # 从前面某个能到达的补给站转移
            if q:
                best = min(best, dp[q[0]] + c_i)

            dp[i] = best

            # 当前点如果可达，就加入单调队列
            if dp[i] < INF:
                while q and dp[q[-1]] >= dp[i]:
                    q.pop()
                q.append(i)

        # 看能否从某个补给站直接冲到终点
        min_cost = INF
        for i in range(m):
            if L - stations[i][0] <= Maxn:
                min_cost = min(min_cost, dp[i])

        ans.append("Yes" if min_cost <= S else "No")

    print("\n".join(ans))

if __name__ == "__main__":
    solve()

## C 最长回文

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

// 使用 unsigned long long 自然溢出，相当于对 2^64 取模，极其高效且免去哈希冲突烦恼
typedef unsigned long long ull;
const ull BASE = 13331; 

// ==========================================
// 工具1：马拉车算法 Manacher (O(N) 极速找原生核心)
// ==========================================
vector<int> manacher(const string& s) {
    int n = s.length();
    string t = "#";
    for (int i = 0; i < n; ++i) {
        t += s[i];
        t += "#";
    }
    int m = t.length();
    vector<int> p(m, 0);
    int c = 0, r = 0;
    for (int i = 0; i < m; ++i) {
        if (r > i) p[i] = min(r - i, p[2 * c - i]);
        while (i - 1 - p[i] >= 0 && i + 1 + p[i] < m && t[i - 1 - p[i]] == t[i + 1 + p[i]]) {
            p[i]++;
        }
        if (i + p[i] > r) {
            c = i;
            r = i + p[i];
        }
    }
    return p;
}

int main() {
    // 【性能法宝】究极 I/O 提速，比赛/笔试必加！
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;
    string A, B;
    cin >> A >> B;

    // ==========================================
    // 工具2：字符串哈希 (O(1) 光速提取指纹)
    // ==========================================
    string revA = A;
    reverse(revA.begin(), revA.end()); // 预处理 A 的反转串，方便倒着比对

    vector<ull> p_pow(n + 1, 1);
    for (int i = 1; i <= n; ++i) p_pow[i] = p_pow[i - 1] * BASE;

    vector<ull> hash_revA(n + 1, 0);
    for (int i = 0; i < n; ++i) hash_revA[i + 1] = hash_revA[i] * BASE + revA[i];

    vector<ull> hash_B(n + 1, 0);
    for (int i = 0; i < n; ++i) hash_B[i + 1] = hash_B[i] * BASE + B[i];

    // 光速比对器：看 A 的左臂和 B 的右臂能匹配多长
    // x 是 A 向左延伸的起点，y 是 B 向右延伸的起点
    auto get_lcp = [&](int x, int y) -> int {
        if (x < 0 || y >= n) return 0;
        int idx_A_rev = n - 1 - x;
        int max_len = min(x + 1, n - y);
        int low = 1, high = max_len, ans = 0;
        
        while (low <= high) {
            int mid = low + (high - low) / 2;
            ull hA = hash_revA[idx_A_rev + mid] - hash_revA[idx_A_rev] * p_pow[mid];
            ull hB = hash_B[y + mid] - hash_B[y] * p_pow[mid];
            if (hA == hB) {
                ans = mid;       // 指纹吻合！记下来并尝试往外继续拉长
                low = mid + 1;
            } else {
                high = mid - 1;  // 失败！步子迈太大了，缩短点
            }
        }
        return ans;
    };

    // 提取所有的回文半径
    vector<int> PA = manacher(A);
    vector<int> PB = manacher(B);

    int max_ans = 0;

    // ==========================================
    // 终极合并：开始“无缝拼接”，找出最长回文
    // ==========================================

    // 情况 1：回文串的核心完全落在 A 中
    for (int i = 0; i < PA.size(); ++i) {
        int rad = PA[i];
        if (rad == 0) continue;
        int L = (i - rad) / 2;
        int R = (i + rad - 2) / 2; // A 的核心是 A[L...R]
        
        // 我们在 R 处切开。左臂从 L-1 向左，右臂从 R 开始在 B 中向右！
        int lcp = get_lcp(L - 1, R);
        int cand = (R - L + 1) + 2 * lcp;
        if (cand > max_ans) max_ans = cand;
    }

    // 情况 2：回文串的核心完全落在 B 中
    for (int i = 0; i < PB.size(); ++i) {
        int rad = PB[i];
        if (rad == 0) continue;
        int L = (i - rad) / 2;
        int R = (i + rad - 2) / 2; // B 的核心是 B[L...R]
        
        // 我们在 L 处切开。左臂从 L 开始在 A 中向左，右臂从 R+1 开始在 B 中向右！
        int lcp = get_lcp(L, R + 1);
        int cand = (R - L + 1) + 2 * lcp;
        if (cand > max_ans) max_ans = cand;
    }

    // 情况 3：核心是空的（偶数回文，中心恰好卡在 A[k] 和 B[k] 之间）
    for (int k = 0; k < n; ++k) {
        int lcp = get_lcp(k, k);
        int cand = 2 * lcp;
        if (cand > max_ans) max_ans = cand;
    }

    // 华丽输出结果
    cout << max_ans << "\n";

    return 0;
}

## D 优惠券

In [ ]:
#include <iostream>
#include <set>
#include <string>
#include <vector>
using namespace std;

const int MAXX = 100000 + 5;

int cnt_[MAXX];   // 当前是否持有该券：0/1
int lastI[MAXX];  // 上一次已知 I x 的位置
int lastO[MAXX];  // 上一次已知 O x 的位置
bool vis[MAXX];

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int m;
    while (cin >> m) {   // 多组数据，直到 EOF
        set<int> unknown;      // 还没被使用的损坏行位置
        vector<int> touched;   // 本组用到过的券编号，便于清空
        touched.reserve(100000);

        int ans = -1;

        for (int i = 1; i <= m; ++i) {
            string op;
            cin >> op;

            // 兼容网页里复制出来的全角“？”
            if (op != "I" && op != "O") {
                if (ans == -1) unknown.insert(i);
                continue;
            }

            int x;
            cin >> x;

            if (!vis[x]) {
                vis[x] = true;
                touched.push_back(x);
                cnt_[x] = 0;
                lastI[x] = 0;
                lastO[x] = 0;
            }

            if (ans != -1) {
                // 已经确定最早错误行了，后面只把输入读完
                if (op == "I") lastI[x] = i;
                else lastO[x] = i;
                continue;
            }

            if (op == "I") {
                ++cnt_[x];

                // 出现连续两次持有，必须在上一次 I 之后找一个 ? 补成 O x
                if (cnt_[x] >= 2) {
                    auto it = unknown.lower_bound(lastI[x]);
                    if (it == unknown.end()) {
                        ans = i;
                    } else {
                        unknown.erase(it);
                        --cnt_[x];
                    }
                }

                lastI[x] = i;
            } else { // op == "O"
                --cnt_[x];

                // 先用后买，必须在上一次 O 之后找一个 ? 补成 I x
                if (cnt_[x] < 0) {
                    auto it = unknown.lower_bound(lastO[x]);
                    if (it == unknown.end()) {
                        ans = i;
                    } else {
                        unknown.erase(it);
                        ++cnt_[x];
                    }
                }

                lastO[x] = i;
            }
        }

        cout << ans << '\n';

        // 清空本组状态
        for (int x : touched) {
            vis[x] = false;
            cnt_[x] = 0;
            lastI[x] = 0;
            lastO[x] = 0;
        }
    }

    return 0;
}

## E 任意点

In [ ]:
#include <iostream>
#include <vector>

using namespace std;

// 帮派名册（存放每个岛屿的大哥是谁）
// 因为 n 最大只有 100，我们开 105 绝对安全
int parent_node[105];

// 核心功能 1：寻找最顶层的“真·大哥” (带路径压缩，极其高效)
int find(int i) {
    if (parent_node[i] == i) {
        return i; // 如果大哥是自己，那自己就是顶层大哥
    }
    // 顺藤摸瓜找大哥，并且直接把自己的上级改成顶层大哥，下次找就快了！
    return parent_node[i] = find(parent_node[i]);
}

// 核心功能 2：两个帮派结盟
void unite(int i, int j) {
    int root_i = find(i);
    int root_j = find(j);
    if (root_i != root_j) {
        parent_node[root_i] = root_j; // i 的大哥 臣服于 j 的大哥
    }
}

int main() {
    // 【习惯性性能加持】虽然数据小，但我们依然挂上 I/O 加速引擎，追求 0 毫秒通关！
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    vector<pair<int, int>> points(n);
    
    // 1. 读取所有岛屿坐标，并初始化帮派名册（每个人一开始都是自己的大哥）
    for (int i = 0; i < n; ++i) {
        cin >> points[i].first >> points[i].second;
        parent_node[i] = i; 
    }

    // 2. 江湖大结盟：两两对比所有岛屿
    for (int i = 0; i < n; ++i) {
        for (int j = i + 1; j < n; ++j) {
            // 如果 x 相同 或 y 相同，说明天然连通，直接合并帮派！
            if (points[i].first == points[j].first || points[i].second == points[j].second) {
                unite(i, j);
            }
        }
    }

    // 3. 清点山头：看看最后到底剩下了几个独立的大哥（连通块）
    int components = 0;
    for (int i = 0; i < n; ++i) {
        if (parent_node[i] == i) {
            components++;
        }
    }

    // 4. 得出结论：需要填海造的人工岛数量 = 帮派总数 - 1
    cout << components - 1 << "\n";

    return 0;
}

## F 通配符匹配

In [ ]:
#include <iostream>
#include <string>
#include <vector>

using namespace std;

// 定义被拆分的零件类型
enum Type { LITERAL, STAR, QUESTION };

// 零件结构体
struct Token {
    Type type;
    string str;     // 如果是纯字母，存放字符串
    int id;         // 给字母纸条编个号
    vector<int> pi; // KMP算法的灵魂：Next数组
};

// 【性能核武】全局二维状态表，防止所有死循环重复计算！
// 10万长度 x 25个最大零件数。采用全局变量直接分配，速度极快！
int visited[100005][25];
bool memo_res[100005][25];
int current_case = 0; // 用例批次号，直接复用状态表，0内存分配耗时！

// 核心状态机（记忆化DFS）
bool dfs(int s_idx, int t_idx, const string& s, const vector<Token>& tokens, const vector<vector<bool>>& matches) {
    // 如果零件用完了，看看嫌疑人名字是不是也刚好查到底了
    if (t_idx == tokens.size()) return s_idx == s.length();
    
    // 如果这个状态以前来过，直接交答案，绝不多算一秒钟！
    if (visited[s_idx][t_idx] == current_case) return memo_res[s_idx][t_idx];

    bool res = false;
    const auto& tok = tokens[t_idx];
    
    if (tok.type == LITERAL) {
        int L = tok.str.length();
        // 直接查 KMP 给出的答案本，不用再一个一个字符去对比了！极其暴力！
        if (s_idx + L <= s.length() && matches[tok.id][s_idx]) {
            res = dfs(s_idx + L, t_idx + 1, s, tokens, matches);
        }
    } else if (tok.type == QUESTION) {
        // '?' 必须强行吃掉一个字符
        if (s_idx < s.length()) {
            res = dfs(s_idx + 1, t_idx + 1, s, tokens, matches);
        }
    } else if (tok.type == STAR) {
     
        // 1. 假装啥也不吃，看后面的
        res = dfs(s_idx, t_idx + 1, s, tokens, matches);
        // 2. 如果不吃不行，那就贪婪地吃掉当前字符，继续保持 '*' 的状态
        if (!res && s_idx < s.length()) {
            res = dfs(s_idx + 1, t_idx, s, tokens, matches);
        }
    }

    // 存入状态表
    visited[s_idx][t_idx] = current_case;
    memo_res[s_idx][t_idx] = res;
    return res;
}

int main() {
    // C++ I/O 狂暴提速
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    string p;
    if (!(cin >> p)) return 0;

    int n;
    if (!(cin >> n)) return 0;

    vector<Token> tokens;
    string cur = "";
    int lit_id = 0;
    
    // 1. 终极拆解法：把通缉令拆成“零件”
    for (char c : p) {
        if (c == '*' || c == '?') {
            if (!cur.empty()) {
                tokens.push_back({LITERAL, cur, lit_id++});
                cur = "";
            }
            if (c == '*') {
                // 连续的 *** 等价于一个 *，极致压缩状态
                if (tokens.empty() || tokens.back().type != STAR) {
                    tokens.push_back({STAR, "*", -1});
                }
            } else {
                tokens.push_back({QUESTION, "?", -1});
            }
        } else {
            cur += c;
        }
    }
    if (!cur.empty()) {
        tokens.push_back({LITERAL, cur, lit_id++});
    }

    // 2. KMP 预处理所有字母零件
    for (auto& tok : tokens) {
        if (tok.type == LITERAL) {
            int m = tok.str.length();
            tok.pi.assign(m, 0);
            for (int i = 1, j = 0; i < m; i++) {
                while (j > 0 && tok.str[i] != tok.str[j]) j = tok.pi[j - 1];
                if (tok.str[i] == tok.str[j]) j++;
                tok.pi[i] = j;
            }
        }
    }

    // 3. 开始暴力核对所有的嫌疑人！
    for (int i = 0; i < n; i++) {
        string s;
        cin >> s;

        current_case++; // 更新批次，全局复用状态表，秒杀清空耗时
        
        // 生成这一轮的 KMP 答案本
        vector<vector<bool>> matches(lit_id, vector<bool>(s.length(), false));

        for (const auto& tok : tokens) {
            if (tok.type == LITERAL) {
                int m = tok.str.length();
                int s_len = s.length();
                for (int idx = 0, j = 0; idx < s_len; idx++) {
                    while (j > 0 && s[idx] != tok.str[j]) j = tok.pi[j - 1];
                    if (s[idx] == tok.str[j]) j++;
                    if (j == m) {
                        matches[tok.id][idx - m + 1] = true; // 记录贴合的坐标
                        j = tok.pi[j - 1];
                    }
                }
            }
        }

        // 丢进状态机进行光速推演
        if (dfs(0, 0, s, tokens, matches)) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
#include <iostream>
#include <vector>
#include <string>

using namespace std;

int main() {
    // 【性能法宝】解除 C++ I/O 封印，让输入输出快如闪电！
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    // 读取 6 种操作的优先级
    vector<string> priority(6);
    for (int i = 0; i < 6; ++i) {
        cin >> priority[i];
    }

    // dp[i][j] 表示：把 i 个盘子的“家族”从柱子 j 整体搬走，最终会落到哪个柱子
    vector<vector<int>> dp(n + 1, vector<int>(3, 0));
    // steps[i][j] 表示：上述整体搬迁一共需要的步数 (注意用 long long 防爆)
    vector<vector<long long>> steps(n + 1, vector<long long>(3, 0));

    // 0代表A，1代表B，2代表C
    
    // ==========================================
    // 基础情况：只有 1 个盘子的时候，直接查优先级表！
    // ==========================================
    for (int start = 0; start < 3; ++start) {
        for (int p = 0; p < 6; ++p) {
            // 找到第一条从 start 柱子出发的规则，就是最高优先级
            if (priority[p][0] - 'A' == start) {
                dp[1][start] = priority[p][1] - 'A';
                steps[1][start] = 1;
                break;
            }
        }
    }

    // ==========================================
    // 开始动态规划：盘子数从 2 推导到 n
    // ==========================================
    for (int i = 2; i <= n; ++i) {
        for (int start = 0; start < 3; ++start) {
            int curr_pile = start;  // n-1 个盘子（子孙）当前所在的位置
            int disk_i = start;     // 最大的盘子（老祖宗）当前所在的位置
            long long total_steps = 0;

            // 1. 子孙们先整体搬走，给老祖宗腾地方
            int next_pile = dp[i - 1][curr_pile];
            total_steps += steps[i - 1][curr_pile];
            curr_pile = next_pile;

            // 2. 老祖宗搬到唯一空着的柱子 (利用 0+1+2=3 的数学魔法求空柱子)
            int empty_peg = 3 - disk_i - curr_pile;
            disk_i = empty_peg;
            total_steps += 1;

            // 3. 子孙们开始追随老祖宗，直到回合！
            while (curr_pile != disk_i) {
                // 子孙们整体搬迁一次
                next_pile = dp[i - 1][curr_pile];
                total_steps += steps[i - 1][curr_pile];
                curr_pile = next_pile;

                if (curr_pile == disk_i) {
                    break; // 追上了，一家团圆！结束这场追逐
                }

                // 如果没追上（子孙占错了柱子），老祖宗只好被迫再去另一个空柱子
                empty_peg = 3 - disk_i - curr_pile;
                disk_i = empty_peg;
                total_steps += 1;
            }

            // 记录下 i 个盘子整体搬迁的最终结果
            dp[i][start] = disk_i;
            steps[i][start] = total_steps;
        }
    }

    // 题目要求：把 n 个盘子从 A 柱 (即 0) 移走的步数
    cout << steps[n][0] << "\n";

    return 0;
}

## H 马步距离

In [ ]:
#include <iostream>
#include <cmath>
#include <algorithm>

using namespace std;

int main() {
    // 【性能法宝】挂载加速引擎，解除 C/C++ I/O 同步，避免大规模读写拖慢速度
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    // 坐标轴可能极大，统一使用 64 位整数防爆
    long long xp, yp, xs, ys;
    
    // 读取输入数据
    if (cin >> xp >> yp >> xs >> ys) {
        
        // 1. 计算两点在 X 轴和 Y 轴上的绝对跨度
        long long dx = abs(xp - xs);
        long long dy = abs(yp - ys);

        // 2. 统一视角：永远让 dx 成为较长的边，极大简化后续判断
        if (dx < dy) {
            swap(dx, dy);
        }

        // 3. 处理无限棋盘上仅有的两个“施展不开”的特例
        if (dx == 1 && dy == 0) {
            cout << 3 << "\n";
        } else if (dx == 2 && dy == 2) {
            cout << 4 << "\n";
        } else {
            // 4. 核心解析解：取“单轴极速”和“综合步幅极速”的最大值
            // 在 C++ 中，(A + B - 1) / B 是对整数除法向上取整的极简写法
            long long steps = max((dx + 1) / 2, (dx + dy + 2) / 3);
            
            // 5. 黑白格奇偶校验：步数的奇偶性必须与空间总位移的奇偶性咬合！
            // 如果不一致，说明必须多花 1 步来调整落点
            if (steps % 2 != (dx + dy) % 2) {
                steps++;
            }
            
            // 输出最终精准答案
            cout << steps << "\n";
        }
    }
    
    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * * @param heights int整型vector 
     * @return int整型
     */
    int largestRectangleArea(vector<int>& heights) {
        int max_area = 0;
        vector<int> st; // 单调栈：这里存放的是柱子的【下标索引】，而不是高度
        int n = heights.size();
        
        // 核心技巧：多遍历一次（i = n）。
        // 相当于在直方图最后面加了一根高度为 0 的“虚拟隐形柱子”。
        // 它的作用是把栈里所有剩余的柱子全部“逼”出来结算面积。
        for (int i = 0; i <= n; ++i) {
            int cur_height = (i == n) ? 0 : heights[i];
            
            // 如果栈不为空，且当前遇到的柱子，比栈顶的柱子还要矮
            // 说明栈顶柱子的【右边界】找到了！开始结算栈顶柱子的面积！
            while (!st.empty() && cur_height < heights[st.back()]) {
                // 1. 抽出栈顶柱子，它的高度就是我们要计算的矩形高度
                int h = heights[st.back()];
                st.pop_back(); 
                
                // 2. 算宽度：右边界是当前索引 i，左边界是弹栈后新的栈顶
                // 如果栈空了，说明刚才弹出的柱子左边没有任何比它矮的，宽度一直延伸到最左边索引0
                int w = st.empty() ? i : (i - st.back() - 1);
                
                // 3. 计算面积并更新最大值
                max_area = max(max_area, h * w);
            }
            
            // 当前柱子的索引入栈，继续保持栈内高度单调递增
            st.push_back(i);
        }
        
        return max_area;
    }
};

## J 消防局的设立

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

// 假设最大基地数不会超过 100005，这在各大厂笔试中是标配安全范围
const int MAXN = 100005;
vector<int> adj[MAXN]; // 族谱记录本（存储每个基地的相连关系）
int up[MAXN];          // 离我最近的路由器距离
int down[MAXN];        // 离我最远的没信号小弟距离
int ans = 0;           // 最终需要建的消防局总数

// 核心遍历函数：从底层往上汇报情况
// u: 当前基地编号, p: 爸爸基地编号
void dfs(int u, int p) {
    int up_val = 1e9;  // 一开始假设底下根本没有路由器 (无穷远)
    int down_val = 0;  // 一开始假设最惨的没信号小弟就是我自己 (距离为0)

    // 先去问问所有儿子的情况
    for (int v : adj[u]) {
        if (v == p) continue; // 别回头问爸爸
        
        dfs(v, u); // 儿子，汇报情况！
        
        // 儿子如果有路由器，我的 up 距离就是儿子的 up + 1
        up_val = min(up_val, up[v] + 1);
        
        // 如果儿子的地盘里有没信号的小弟，我就得记录下来
        if (down[v] != -1) {
            down_val = max(down_val, down[v] + 1);
        }
    }

    // 【神奇的化学反应】
    // 如果底下最近的路由器，它的信号 (up_val) 还能跑够路程去覆盖底下最惨的小弟 (down_val)
    if (up_val + down_val <= 2) {
        down_val = -1; // 皆大欢喜，小弟有救了，危险解除！
    }

    // 【不能再忍了】
    // 没信号的小弟离我已经 2 步远了，再往上汇报就来不及了！
    if (down_val == 2) {
        ans++;         // 必须在 u 这里建一个消防局！
        up_val = 0;    // u 自己变成了路由器，距离为 0
        down_val = -1; // 危险彻底解除
    }

    // 记录好本本，准备向爸爸汇报
    up[u] = up_val;
    down[u] = down_val;
}

int main() {
    // 【性能法宝】解除 C/C++ I/O 同步，防止 10 万级别数据读写超时！
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0; // 读取基地总数

    // 读入 n-1 条道路记录
    // 第 i 行 (i 从 2 开始) 记录着 i 到 a[i] 之间有一条道路
    for (int i = 2; i <= n; ++i) {
        int u;
        cin >> u;
        adj[i].push_back(u);
        adj[u].push_back(i); // 道路是双向的
    }

    // 从根节点 1 开始，作为最初的起点向下探索
    // 1 没有爸爸，所以设爸爸为 0
    dfs(1, 0);

    // 【最后的扫尾】
    // 如果全都汇报完了，根节点 1 发现自己或者底下的某个兄弟竟然还没信号
    // (只要 down >= 0，说明还有隐患)，只能在根节点补建一个兜底！
    if (down[1] >= 0) {
        ans++;
    }

    // 华丽输出结果
    cout << ans << "\n";

    return 0;
}our code here